### dependencies & globals


In [1]:
import requests
import pandas as pd
import geopandas as gpd
from datetime import datetime
import uuid

county_selection = [
    "Fulton",
    "Cobb",
    "DeKalb",
    "Fayette",
    "Gwinnett",
    "Cherokee",
    "Douglas",
    "Forsyth",
    "Clayton",
    "Henry",
    "Rockdale"
]

### API call to get GDOT projects


In [ ]:
# Define the base URL of the MapServer layer
base_url = "https://rnhp.dot.ga.gov/hosting/rest/services/GDOT_Public_Outreach/Hub_Project_Search/MapServer/2/query"

# Define query parameters
params = {
    # filter by construction status to include both under construction and pre-construction
    "where": "CONSTRUTION_STATUS_DERIVED = 'PRE-CONSTRUCTION'",
    "outFields": "*",  # Get all fields
    "f": "geojson",  # Request data in GeoJSON format
    "returnGeometry": "true",  # Include geometry data
    "resultOffset": 0,  # Start from the first record
    "resultRecordCount": 1000  # Get 1000 records
}

# list to store each batch of GeoDataFrames
gdfs = []

while True:

    # Make the GET request
    response = requests.get(base_url, params=params)

    # Check if the request was successful
    if response.status_code == 200:
        data = response.json()

        batch_gdf = gpd.GeoDataFrame.from_features(data["features"])

        # if no more data is returned, break the loop
        if batch_gdf.empty:
            break

        gdfs.append(batch_gdf)

        print(
            f"Fetched {len(batch_gdf)} records (offset: {params['resultOffset']})")

        # increase offset by the batch size to fetch the next chunk
        params["resultOffset"] += params["resultRecordCount"]

    else:
        print(
            f"❌ Error fetching data: {response.status_code} - {response.text}")
        break

# Filter out empty GeoDataFrames
gdfs = [gdf.dropna(axis=1, how='all') for gdf in gdfs if not gdf.empty]

# Combine all batches into a single GeoDataFrame
if gdfs:
    full_gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))

else:
    print("❌ No data retrieved.")

# drop any columns that ONLY contain NaN values
full_gdf = full_gdf.dropna(axis=1, how="all")

# replace all-caps values in the 'CONSTRUTION_STATUS_DERIVED' column
full_gdf['CONSTRUTION_STATUS_DERIVED'] = full_gdf['CONSTRUTION_STATUS_DERIVED'].str.replace(
    'PRE-CONSTRUCTION', 'Pre-Construction')

# if the 'CONTRACT_DESCRIPTION' column is empty, replace it with the 'SHORT_DESCR' column
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].fillna(
    full_gdf['SHORT_DESCR'])

# define a list of sub-strings to keep in all caps; everything else in the 'SHORT_DESCR' column should be converted to title case
keep_all_caps = ["SR ", "CO ", "CR ", "US ", "RD", "SO ",
                 "CS ", "MI ", "SE ", "NE ", "SW ", "NW ",
                 "NS ", "CSX", "CCTV", "-TIA", " - TIA", "LCI", "GRTA"]

full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.title()
for sub_str in keep_all_caps:
    full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
        sub_str.title(), sub_str)

full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Brdg", "Bridge")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Mtn ", "Mountain ")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Phase Ii", "Phase II")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Ph Ii", "Phase II")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Phase IIi", "Phase III")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Ph IIi", "Phase III")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Phase Iv", "Phase IV")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Phase Vi", "Phase VI")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Resf ", "Resurface ")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Rsrf ", "Resurface ")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Shldr ", "Shoulder ")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Recnst", "Reconstruction")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Cnst ", "Construction ")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Fm ", "From ")


# filter out records that have either 'WPH' (completed) or 'REJ' (rejected) in the 'CONST_STAT_CD' column
full_gdf = full_gdf[~full_gdf['CONST_STAT_CD'].isin(['WPH', 'REJ'])]

# rename columns
full_gdf = full_gdf.rename(columns={
    'LAST_REFRESH_DTTM': 'Last_refresh',
    'PROJECT_COUNTIES': 'Project_counties',
    'PROJ_ID': 'Project_ID',
    'CONTRACT_DESCRIPTION': 'Project_description',
    'CONTRACTOR_NAME': 'Contractor_name',
    'IS_TIA_PROJECT': 'Is_TIA_project',
})

# drop unneeded columns
full_gdf = full_gdf.drop(columns=[
    'OBJECTID',
    'PRIORITY_CD',
    'SOURCE_OF_CONSTRUCTION_DATES',
    'CONTRACT_ID',
    'CONSTRUTION_STATUS_DERIVED_RSN',
    'PAYMENT_PERCENT_COMPLETE',
    'ESRI_OID',
    'SHORT_DESCR',
    'REC_STATUS',
    'LET_RESP_CD',
    'PRIORITY_CD_DESCR',
    'CONST_STAT_CD',
    'CONSTRUCTION_PERCENT_COMPLETE',
    'CONSTRUTION_STATUS_DERIVED',
    'CURR_COMPLETION_DATE',
    'AWARD_DATE'
])

# create URL column
full_gdf['Project_URL'] = 'https://www.dot.ga.gov/applications/geopi/Pages/Dashboard.aspx?ProjectId=' + \
    full_gdf['Project_ID'].astype(str)

# Get the current date and time
current_date = datetime.datetime.now().strftime("%Y-%m-%d")

# Add a new property to the GeoJSON feature collection
full_gdf["data_refreshed"] = current_date

# Add a unique ID column
full_gdf["feature_id"] = [str(uuid.uuid4()) for _ in range(len(full_gdf))]

# Ensure the ID is stored as a string for compatibility with GeoJSON
full_gdf["feature_id"] = full_gdf["feature_id"].astype(str)

# geographic filter: just get those in the metro
full_gdf = full_gdf[full_gdf['Project_counties'].str.contains(
    '|'.join(county_selection),
    na=False)]

# export full_gdf to a geojson file inside the 'data' folder
full_gdf.to_file("data/GDOT_export.geojson", driver="GeoJSON")
csv_export = full_gdf.drop(columns=['geometry']).to_csv(
    'GDOT_export.csv', index=False)

# print status
print(f"✅ Successfully exported {len(full_gdf):,} records!")

# export a text file to 'data' containing today's date
current_date = datetime.now().strftime("%B %d, %Y")
with open("data/current_date.txt", "w") as f:
    f.write(current_date)

# Display final GeoDataFrame
full_gdf

Fetched 1000 records (offset: 0)
Fetched 1000 records (offset: 1000)
Fetched 668 records (offset: 2000)
✅ Successfully exported 481 records!


/opt/homebrew/Caskroom/miniconda/base/envs/research/lib/python3.11/site-packages/pyogrio/geopandas.py:662: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


,geometry,Project_ID,Project_counties,Is_TIA_project,Project_description,Last_refresh,Project_URL,data_refreshed,feature_id
3,"LINESTRING (-84.3719 33.53515, -84.37275 33.53...",M006699,Clayton,No,SR 138 From Fulton County Line To North Ave,1740567326000,https://www.dot.ga.gov/applications/geopi/Page...,2025-02-26,f9e35bda-31ef-48b1-bebd-7bdb50020416
5,"LINESTRING (-84.46072 33.89032, -84.46105 33.8...",M006701,Cobb,No,I-75 From I-285 To N Of Hickory Grove Road - E...,1740567326000,https://www.dot.ga.gov/applications/geopi/Page...,2025-02-26,af897a32-84c2-4b15-a6dd-781c80dffd44
9,"LINESTRING (-84.52295 33.96504, -84.5232 33.96...",M006706,Cobb,No,I-75 Nb Noise Walls @ 6 Locs In Cobb County,1740567326000,https://www.dot.ga.gov/applications/geopi/Page...,2025-02-26,ccc07253-380b-48ff-9365-607477e3e9d7
10,"LINESTRING (-84.49085 33.93189, -84.49126 33.9...",M006707,Cobb,No,I-75 Nb Noise Walls @ 7 Locs In Cobb County,1740567326000,https://www.dot.ga.gov/applications/geopi/Page...,2025-02-26,63bc3784-0063-47c7-bfd5-8de281bb87ef
11,"LINESTRING (-84.55172 33.99141, -84.55213 33.9...",M006708,Cobb,No,I-75 Nb Noise Walls @ 9 Locs In Cobb County,1740567326000,https://www.dot.ga.gov/applications/geopi/Page...,2025-02-26,fd167353-4ec5-4103-b910-be2b3642cc15
...,...,...,...,...,...,...,...,...,...
2622,"LINESTRING (-84.07217 33.69695, -84.07185 33.6...",0013566,"DeKalb , Rockdale",No,Old Covington Hwy From SR 124 To CR 67/Lake Ca...,1740567326000,https://www.dot.ga.gov/applications/geopi/Page...,2025-02-26,30052c37-1de0-4bdc-b010-7456583bc214
2630,"LINESTRING (-84.09287 34.25212, -84.09103 34.2...",0013571,Forsyth,No,SR 306 From SR 400 To SR 369,1740567326000,https://www.dot.ga.gov/applications/geopi/Page...,2025-02-26,e7eda99b-16eb-498a-84fd-5d1c254e79fd
2657,"LINESTRING (-84.00056 33.68336, -84.00084 33.6...",0013594,Rockdale,No,SR 20 & Sigman Road From CS 442/Irwin Bridge R...,1740567326000,https://www.dot.ga.gov/applications/geopi/Page...,2025-02-26,b26e6e77-1518-4b73-bde9-803eb5bd1042
2659,"LINESTRING (-84.64949 33.79647, -84.64877 33.7...",0013733,Douglas,No,SR 5/US 78 @ SR 6/US 278,1740567326000,https://www.dot.ga.gov/applications/geopi/Page...,2025-02-26,5c4666a7-817d-4086-b45a-8d28341a645e


### misc cleaning
